# Notebook 7 – Encoding Categorical Variables

## 1. Nominal, ordinal, and binary variables
Nominal categories have no natural order (region); ordinal categories have an order (low/medium/high); binary variables have two states (yes/no). Encoding should preserve the meaning of the category.

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

df = pd.DataFrame({
    "Region": ["North", "South", "East", "North"],
    "Satisfaction": ["Low", "High", "Medium", "Low"],
    "Churn": ["No", "Yes", "No", "No"]
})
display(df)

,Region,Satisfaction,Churn
0,North,Low,No
1,South,High,Yes
2,East,Medium,No
3,North,Low,No


## 2. Label encoding and ordinal encoding
Label encoding maps labels to integers; it is commonly used for target labels, not nominal input features where numbers imply an artificial order. Ordinal encoding is suitable when a meaningful order is explicitly defined.

In [2]:
label_encoder = LabelEncoder()
target_encoded = label_encoder.fit_transform(df["Churn"])
ordinal_encoder = OrdinalEncoder(categories=[["Low", "Medium", "High"]])
satisfaction_encoded = ordinal_encoder.fit_transform(df[["Satisfaction"]]).ravel()
print("Target classes:", label_encoder.classes_.tolist())
print("Encoded target:", target_encoded.tolist())
print("Ordinal satisfaction:", satisfaction_encoded.tolist())

Target classes: ['No', 'Yes']
Encoded target: [0, 1, 0, 0]
Ordinal satisfaction: [0.0, 2.0, 1.0, 0.0]


**Output explanation:** The target encoder maps class labels to integer codes. The ordinal encoder follows the explicitly supplied Low < Medium < High order.

## 3. One-hot encoding and dummy variables
One-hot encoding creates one binary column per category. Dummy encoding commonly drops one category as a reference, often useful in linear models with an intercept. High-cardinality categories can create many columns.

In [3]:
one_hot = pd.get_dummies(df["Region"], prefix="Region", dtype=int)
dummy = pd.get_dummies(df["Region"], prefix="Region", drop_first=True, dtype=int)
display(one_hot)
display(dummy)

,Region_East,Region_North,Region_South
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0


,Region_North,Region_South
0,1,0
1,0,1
2,0,0
3,1,0


**Output explanation:** One-hot encoding creates a column for each category; `drop_first=True` omits one reference category. Tree-based models can use different encodings, so choose based on model and feature meaning.

## 4. Frequency encoding and target encoding
Frequency encoding replaces categories with their frequency. Target encoding uses a target statistic per category and can leak target information unless implemented with training-only and out-of-fold safeguards. It may overfit rare categories.

In [4]:
freq = df["Region"].map(df["Region"].value_counts(normalize=True))
print("Frequency encoding:", freq.tolist())

# Illustrative target means; do not use this naive calculation before train/test split.
target_num = df["Churn"].map({"No": 0, "Yes": 1})
target_mean = df.assign(target_num=target_num).groupby("Region")["target_num"].mean()
print("Illustrative category target means:", df["Region"].map(target_mean).tolist())

Frequency encoding: [0.5, 0.25, 0.25, 0.5]
Illustrative category target means: [0.0, 1.0, 0.0, 0.0]


**Output explanation:** Frequency encoding represents how common each category is. The target mean example is only illustrative; in real modeling, use a leakage-safe target encoder and fit it on training data only.

## 5. Unknown categories and high cardinality
A production model may receive a category not seen during training. Configure encoders such as `OneHotEncoder(handle_unknown="ignore")`. High-cardinality fields may require grouping rare values or carefully validated alternative encoding.